# ASL Fingerspelling: Sign Language MNIST Template

<a href="https://colab.research.google.com/github/ML-Rice/asl_fingerspelling_classifier/blob/main/mnist_template.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In this notebook you will build a complete machine learning pipeline, from raw data to a trained and evaluated model. Everyone on the team follows the same steps, but each person builds a **different model**. At the end we compare results as a team.

### How to use this notebook
1. In Colab: **File → Save a copy in Drive**, then rename your copy to `<yourname>_mnist.ipynb`.
2. (Optional for MNIST) **Runtime → Change runtime type → T4 GPU**. MNIST is small, so CPU works too.
3. Run the **Setup** cells. They are already written for you.
4. Work through **Steps 1 to 6**. Every cell with a `# TODO` or a comment and no code is yours to write.
5. Stuck for more than ~30 minutes? Check the docs linked in each step, search the error message, then ask in the group chat.

### The pipeline
```
Load data → Explore (EDA) → Prepare data → Build model → Train + validate → Evaluate on test → Compare
  (given)      Step 1          Step 2         Step 3         Step 4             Step 5          Step 6
```

### Team rules (so our results can be compared)
- Use the train/validation split **exactly** as written in Step 2.4.
- Don't touch the test set until Step 5.
- Report your results in the format of Step 5.6.

---
## Setup (already done for you, just run these cells)

In [ ]:
# Install kagglehub if it's missing (it's preinstalled on Colab, so this is quick).
# You can ignore a "restart the kernel" message.
%pip install -q kagglehub

In [ ]:
# All the libraries you need for this project
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

import kagglehub

# Seeds make random numbers the same every run, so results are reproducible
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

# Use a GPU if one is available
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():  # Apple Silicon Macs
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print("PyTorch", torch.__version__, "| device:", device)

### Load the data
**Sign Language MNIST** contains 28×28 grayscale images of a hand signing letters of the alphabet.
Each row of the CSV is **one image**: a `label` column, followed by 784 pixel columns (28 × 28 = 784).

[Dataset page on Kaggle](https://www.kaggle.com/datasets/datamunge/sign-language-mnist)

In [ ]:
# Downloads the dataset from Kaggle (only the first time; after that it's cached)
data_path = Path(kagglehub.dataset_download("datamunge/sign-language-mnist"))

train_df = pd.read_csv(data_path / "sign_mnist_train.csv")
test_df = pd.read_csv(data_path / "sign_mnist_test.csv")

print("Train:", train_df.shape)
print("Test: ", test_df.shape)

---
## Step 1: Explore the data (EDA)
Before building any model, you need to understand your data: what it looks like, how much there is, and whether anything is unusual. Skipping this step is the most common beginner mistake.

**Useful tools:** pandas `.head()`, `.shape`, `.describe()`, `.value_counts()` · matplotlib `plt.imshow()`, `plt.subplots()`, `plt.bar()`
**Docs:** [pandas: 10 minutes to pandas](https://pandas.pydata.org/docs/user_guide/10min.html) · [matplotlib: imshow](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.imshow.html)

In [ ]:
# 1.1 Look at the first few rows of train_df.
# What are the columns? What does one row represent?


In [ ]:
# 1.2 How many rows and columns do train_df and test_df have?
# Print general statistics of the table (hint: .describe()).
# What are the minimum and maximum pixel values?


In [ ]:
# 1.3 How many images of each label are in the training set? Plot it as a bar chart.
# Is the dataset balanced (about the same number of images per label)?
# Which label numbers are missing, and why? (hint: label 0 = A, 1 = B, 2 = C, ...)


In [ ]:
# 1.4 Take one row: separate the label from the 784 pixel values,
# reshape the pixels into a 28x28 array, and show it with plt.imshow(..., cmap="gray").
# Put the letter it shows in the plot title.


In [ ]:
# 1.5 Make a grid with one example image for every letter (24 letters).
# hint: fig, axes = plt.subplots(4, 6, figsize=(12, 8))


In [ ]:
# 1.6 Compare the label counts of train_df and test_df.
# Do the train and test sets look similar?


### Step 1 questions
*Double-click this cell to write your answers.*

1. How many training images and test images are there?
2. Is the dataset balanced?
3. Which letters are missing, and why?
4. Which letters look similar to each other? Which ones do you predict your model will confuse?

---
## Step 2: Prepare the data
A model can't read a pandas DataFrame. In this step you turn the table into PyTorch **tensors**, split off a **validation set**, and wrap everything in **DataLoaders** that feed the model small batches of images.

**Why a validation set?** We train on the training set. After each epoch we check the model on images it has **never trained on** (validation). That tells us whether it is actually learning or just memorizing. The **test set** stays locked away until the very end (Step 5).

**Docs:** [train_test_split](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html) · [PyTorch: Datasets & DataLoaders](https://pytorch.org/tutorials/beginner/basics/data_tutorial.html)

In [ ]:
# 2.1 Split train_df into:
#   X -> the pixel values (every column except "label") as a NumPy array
#   y -> the labels as a NumPy array
# Do the same for test_df to get X_test and y_test.


In [ ]:
# 2.2 Fix the labels.
# Labels go from 0 to 24 but skip 9 (J), so there are only 24 classes.
# PyTorch expects labels 0, 1, ..., 23 with no gaps.
# Shift every label above 9 down by 1 (hint: np.where), for both y and y_test.
# Check: the unique labels should now be exactly 0 to 23.

LETTERS = list("ABCDEFGHIKLMNOPQRSTUVWXY")  # LETTERS[new_label] gives the letter (no J or Z)


In [ ]:
# 2.3 Scale the pixel values from 0-255 down to 0-1 (divide by 255.0), for X and X_test.
# Why do you think this helps training?


In [ ]:
# 2.4 Split X and y into a training set and a validation set with train_test_split.
# Everyone on the team must use EXACTLY these settings so our results are comparable:
#     test_size=0.15, random_state=42, stratify=y
# Name the results X_train, X_val, y_train, y_val, and print the shape of each.


### Input shape depends on your model
Each model type expects images in a different shape:

| Model | Shape of one batch | What it means |
|---|---|---|
| `logistic`, `mlp` | `(batch, 784)` | Each image is one flat list of 784 numbers |
| `cnn1d` | `(batch, 1, 784)` or `(batch, 28, 28)` | A sequence: `(channels, length)`. Try both and see which works better |
| `cnn2d` | `(batch, 1, 28, 28)` | An image: `(channels, height, width)`. 1 channel because it's grayscale |

In [ ]:
# 2.5 Convert X_train, X_val, X_test into float32 tensors, and y_train, y_val, y_test into long (int64) tensors.
# hint: torch.tensor(X_train, dtype=torch.float32)
# Then reshape the X tensors into the shape your model needs (see the table above).
# hint: .reshape(-1, 1, 28, 28), where -1 means "figure out this dimension for me"


In [ ]:
# 2.6 Wrap the tensors in TensorDatasets, then in DataLoaders:
#   train_loader -> shuffle=True (why should training data be shuffled?)
#   val_loader, test_loader -> shuffle=False

BATCH_SIZE = 64


In [ ]:
# 2.7 Sanity check: take one batch from train_loader and print the shapes.
# hint: images, labels = next(iter(train_loader))
# Expected: images -> (64, <your model's input shape>), labels -> (64,)


---
## Step 3: Build the model
In PyTorch a model is a class that inherits from `nn.Module` and has two parts:
- `__init__`: **create** the layers
- `forward(x)`: **use** the layers, describing how data flows through them

**Rules for every model on the team:**
- The output shape must be `(batch, 24)`: one score (called a *logit*) per letter.
- Do **not** add a softmax at the end. The loss function in Step 4 does that for you.

| Model | Idea | Layers to look up |
|---|---|---|
| `logistic` | Logistic (softmax) regression: a single linear layer, no hidden layers. Our **baseline** | `nn.Linear` |
| `mlp` | Multi-layer perceptron: several linear layers with activation functions between them | `nn.Linear`, `nn.ReLU`, `nn.Dropout` |
| `cnn1d` | 1D convolutional network: slides filters along the image as if it were a sequence | `nn.Conv1d`, `nn.MaxPool1d`, `nn.Flatten`, `nn.Linear` |
| `cnn2d` | 2D convolutional network: slides small filters over the image to learn shapes such as edges and fingers | `nn.Conv2d`, `nn.MaxPool2d`, `nn.Flatten`, `nn.Linear` |

**Docs:** [PyTorch: Build the Neural Network](https://pytorch.org/tutorials/beginner/basics/buildmodel_tutorial.html) · [torch.nn layers](https://pytorch.org/docs/stable/nn.html)

In [ ]:
# 3.1 Define your model. Fill in __init__ (create layers) and forward (use them).

class MyModel(nn.Module):
    def __init__(self, num_classes=24):
        super().__init__()
        # TODO: create your layers here

    def forward(self, x):
        # TODO: pass x through your layers and return the result
        return x


In [ ]:
# 3.2 Create your model and move it to the device: model = MyModel().to(device)
# Print the model to see its layers.
# Count its trainable parameters (hint: sum(p.numel() for p in model.parameters()))


In [ ]:
# 3.3 Sanity check: pass the batch from 2.7 through your model (move it to device first).
# The output shape must be (64, 24). If it isn't, fix your model or your reshape in 2.5.


---
## Step 4: Train and validate
Training repeats the same loop many times:
**show the model a batch → it predicts → measure how wrong it was (loss) → compute gradients (backward) → adjust the weights a little (optimizer step)**

- **Epoch:** one full pass over all the training data.
- After every epoch, measure the **validation** loss and accuracy. No learning happens here; we only check.
- Whenever validation accuracy beats the best so far, save a **checkpoint** (`best.pt`). Then, if the model gets worse later (overfitting), we still have its best version.

**Key functions:** `model.train()` / `model.eval()` · `torch.no_grad()` · `optimizer.zero_grad()` · `loss.backward()` · `optimizer.step()`
**Docs:** [PyTorch: Optimizing Model Parameters](https://pytorch.org/tutorials/beginner/basics/optimization_tutorial.html)

In [ ]:
# 4.1 Hyperparameters are settings YOU choose; the model doesn't learn them.
# Start with these values, and experiment in Step 6.
LEARNING_RATE = 1e-3
EPOCHS = 10

# TODO: create the loss function (hint: nn.CrossEntropyLoss())
# TODO: create the optimizer (hint: torch.optim.Adam(model.parameters(), lr=LEARNING_RATE))


In [ ]:
# 4.2 Write a function train_one_epoch(model, loader, loss_fn, optimizer)
# that returns (average loss, accuracy) for one epoch.
#
#   1. model.train()
#   2. loop over (images, labels) in loader:
#        a. move images and labels to the device (.to(device))
#        b. optimizer.zero_grad()           -> clear old gradients
#        c. outputs = model(images)         -> forward pass
#        d. loss = loss_fn(outputs, labels) -> how wrong were we?
#        e. loss.backward()                 -> compute gradients
#        f. optimizer.step()                -> update the weights
#        g. add up the loss and the number of correct predictions
#           (hint: predictions = outputs.argmax(dim=1))
#   3. return total_loss / number_of_batches, correct / number_of_images


In [ ]:
# 4.3 Write a function evaluate(model, loader, loss_fn) that returns (average loss, accuracy).
# It's the same as train_one_epoch, except:
#   - model.eval() instead of model.train()
#   - wrap the loop in `with torch.no_grad():`
#   - NO zero_grad, backward or step: we are only measuring, not learning


In [ ]:
# 4.4 The training loop. For each epoch:
#   1. call train_one_epoch on train_loader
#   2. call evaluate on val_loader
#   3. record the 4 numbers in a history dict:
#        history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
#   4. if val accuracy is the best so far, save a checkpoint:
#        torch.save(model.state_dict(), "best.pt")
#   5. print the epoch number, train loss/acc and val loss/acc
# After the loop, also save the final model to "last.pt".
# Bonus: measure the total training time (hint: time.perf_counter()).


---
## Step 5: Evaluate the model
Now we find out how good the model really is.

In [ ]:
# 5.1 Plot the training curves from history, side by side:
#   left:  train loss and val loss per epoch
#   right: train accuracy and val accuracy per epoch
# Is your model overfitting? (Train keeps improving while val stops improving or gets worse.)


In [ ]:
# 5.2 Load the best checkpoint back into your model:
#   model.load_state_dict(torch.load("best.pt"))
# Why do we use best.pt instead of the model as it is after the last epoch?


### The test set
This is the **first time** we use the test set. Run it only when you're happy with your model. If you keep tweaking until the test accuracy goes up, the test set stops being "unseen" data and its number becomes meaningless.

In [ ]:
# 5.3 Evaluate the best model on test_loader and print the test loss and accuracy.
# How does it compare with your best validation accuracy?


In [ ]:
# 5.4 Confusion matrix on the test set:
#   1. collect every prediction and true label from test_loader (model.eval() + torch.no_grad())
#   2. cm = confusion_matrix(all_labels, all_preds)
#   3. plot it: sns.heatmap(cm, xticklabels=LETTERS, yticklabels=LETTERS), with figsize around (12, 10)
#      and axis labels "Predicted" and "True"
#   4. print classification_report(all_labels, all_preds, target_names=LETTERS)
# Which letters get confused the most? Did you predict that in Step 1?


In [ ]:
# 5.5 Inference speed: how long does your model take to predict ONE image?
#   1. model.eval(), and take a single test image with shape (1, <your input shape>)
#   2. warm up: run it through the model ~10 times and ignore those timings
#   3. inside torch.no_grad(), time 1000 predictions with time.perf_counter()
#      (on a GPU, call torch.cuda.synchronize() before reading the time)
#   4. print the average milliseconds per image
# This matters later: live webcam predictions must keep up with video (~30 frames per second).


In [ ]:
# 5.6 Your results. Fill in the values and run the cell. We will compare everyone's results at the meeting.
results = {
    "model": None,         # TODO: "logistic", "mlp", "cnn1d" or "cnn2d"
    "num_params": None,    # TODO: from 3.2
    "best_val_acc": None,  # TODO: from 4.4
    "test_acc": None,      # TODO: from 5.3
    "ms_per_image": None,  # TODO: from 5.5
    "train_time_s": None,  # TODO: from 4.4 (bonus)
    "epochs": EPOCHS,
    "learning_rate": LEARNING_RATE,
    "batch_size": BATCH_SIZE,
}
print(results)

---
## Step 6: Reflect and experiment
*Double-click this cell to write your answers.*

1. What is your final test accuracy? Is it close to your validation accuracy?
2. Is your model overfitting? Point to your curves to explain how you know.
3. Which letters does your model confuse, and why might that be?
4. Why do you think your model type did better or worse than the others? (We'll discuss this as a team.)

### Experiment log
Change **one thing at a time**, re-run Steps 3 to 5, and record what happened:

| Run | What I changed | Best val acc | Test acc | Notes |
|---|---|---|---|---|
| 1 | Baseline (lr=1e-3, 10 epochs, batch 64) | | | |
| 2 | | | | |
| 3 | | | | |

**Ideas:** learning rate (1e-2, 1e-4), more epochs, batch size, bigger or smaller layers, `nn.Dropout`, `nn.BatchNorm`.
**Stretch goals:** data augmentation (small rotations and shifts), a learning-rate scheduler, early stopping.

---
## Submitting
1. **Runtime → Restart session and run all**. The notebook must run from top to bottom without errors.
2. **File → Download → Download .ipynb**.
3. On the GitHub repo page: **Add file → Upload files**, drag in your `<yourname>_mnist.ipynb`, then click **Commit changes**.